# Phase 1

In [1]:
import json
import hashlib
import numpy as np

# Load the Phase 1 registry
with open('data/prepared_smoke/registry.json') as f:
    reg = json.load(f)

key = "synth_shift"
meta = reg['datasets'][key]
data = np.load(f'data/prepared_smoke/{key}.npz')

print("=== PHASE 1 LOGIC CHECK ===")
print(f"Dataset: {key}")

# 1. Check the digest matches the actual saved validation indices
saved_val_idx = data['val_idx']
computed_digest = hashlib.sha256(saved_val_idx.tobytes()).hexdigest()

print(f"1. Validation Digest Match: {computed_digest == meta['val_digest']}")

# 2. Check that validation indices are sorted (deterministic)
print(f"2. Validation Indices Sorted: {np.all(saved_val_idx[:-1] <= saved_val_idx[1:])}")

# 3. Check that the validation set is exactly 20% of the train+val set
total_train_val = meta['n_train'] + meta['n_val']
expected_val_size = int(total_train_val * 0.2)
print(f"3. Validation Split Ratio Correct: {meta['n_val']} == {expected_val_size}")

# 4. Check that X_val and y_val match the length of val_idx
print(f"4. X_val shape matches val_idx: {data['X_val'].shape[0] == len(saved_val_idx)}")
print(f"5. y_val shape matches val_idx: {data['y_val'].shape[0] == len(saved_val_idx)}")

=== PHASE 1 LOGIC CHECK ===
Dataset: synth_shift
1. Validation Digest Match: True
2. Validation Indices Sorted: True
3. Validation Split Ratio Correct: 150000 == 150000
4. X_val shape matches val_idx: True
5. y_val shape matches val_idx: True


# Phase 2

In [2]:
data = np.load('data/prepared_smoke/synth_shift.npz')
val_idx_global = set(data['val_idx'].tolist())
train_idx_global = data['train_idx']

with open('data/prepared_smoke/subsample_registry.json') as f:
    sub_reg = json.load(f)

print("\n=== PHASE 2 LOGIC CHECK (FIXED) ===")
entry = sub_reg["synth_shift_vol10000_seed0"]
local_idx = np.array(entry['local_indices'], dtype=np.int64)
global_idx = np.array(entry['global_indices'], dtype=np.int64)

print(f"1. Size correct:               {len(local_idx) == entry['n_samples']}")
print(f"2. Digest matches:             "
      f"{hashlib.sha256(local_idx.tobytes()).hexdigest() == entry['indices_digest']}")
print(f"3. Local -> Global mapping ok: "
      f"{np.array_equal(global_idx, train_idx_global[local_idx])}")
print(f"4. NO val contamination:       "
      f"{len(set(global_idx.tolist()).intersection(val_idx_global)) == 0}")
print(f"5. Local indices in range:     "
      f"{local_idx.max() < len(train_idx_global)}")

# Determinism
rng = np.random.default_rng(0)
expected = rng.choice(np.arange(len(train_idx_global)),size=10000, replace=False).astype(np.int64)
expected.sort()
print(f"6. Deterministic:              {np.array_equal(local_idx, expected)}")

# Also verify audit trail of Phase 1
print("\n=== PHASE 1 AUDIT ===")
print(f"train_idx digest matches meta: "
      f"{hashlib.sha256(train_idx_global.tobytes()).hexdigest() == json.load(open('data/prepared_smoke/registry.json'))['datasets']['synth_shift']['train_digest']}")
print(f"train/val disjoint:            "
      f"{len(set(train_idx_global.tolist()).intersection(val_idx_global)) == 0}")
print(f"train + val == original size:  "
      f"{len(train_idx_global) + len(val_idx_global)}")


=== PHASE 2 LOGIC CHECK (FIXED) ===
1. Size correct:               True
2. Digest matches:             True
3. Local -> Global mapping ok: True
4. NO val contamination:       True
5. Local indices in range:     True
6. Deterministic:              True

=== PHASE 1 AUDIT ===
train_idx digest matches meta: True
train/val disjoint:            True
train + val == original size:  750000


In [3]:
with open('data/prepared_smoke/subsample_registry.json') as f:
    reg = json.load(f)

# Volume ladder per dataset
for ds in ["synth_shift", "synth_iid"]:
    vols = sorted({v['volume_level'] for k, v in reg.items()
                   if v['dataset_key'] == ds})
    print(f"{ds}: {vols}")

synth_shift: [10000, 30000, 100000, 300000]
synth_iid: [10000, 30000, 100000, 300000]


# The Ultimate "Same Indices" Guarantee (Simulating Phase 3 & 4)

In [4]:
import json
import numpy as np

print("\n=== PHASE 3/4 SIMULATION ===")

# 1. Load the raw prepared data (Phase 1 output)
data = np.load('data/prepared_smoke/synth_shift.npz')
X_train_full = data['X_train']
y_train_full = data['y_train']

# 2. Load the subsample registry (Phase 2 output)
with open('data/prepared_smoke/subsample_registry.json') as f:
    sub_reg = json.load(f)

# 3. Both branches look up the SAME key
experiment_key = "synth_shift_vol10000_seed0"
entry = sub_reg[experiment_key]

# 4. Both branches slice the data using the EXACT SAME local indices
local_idx = entry['local_indices']
X_subset = X_train_full[local_idx]
y_subset = y_train_full[local_idx]

print(f"Experiment: {experiment_key}")
print(f"Boosting Branch (XGBoost) sees:    X shape {X_subset.shape}, y shape {y_subset.shape}")
print(f"Foundation Branch (TabPFN) sees:   X shape {X_subset.shape}, y shape {y_subset.shape}")

# 5. Prove they are identical
identical = np.array_equal(X_subset, X_train_full[local_idx])
print(f"Are the training sets identical across branches? {identical}")

# 6. Prove that the validation and test sets remain completely untouched
X_val = data['X_val']
X_id  = data['X_id']
print(f"Validation set is fixed: {X_val.shape[0] == len(data['val_idx'])}")
print(f"ID Test set is fixed:    {X_id.shape[0] == data['y_id'].shape[0]}")

# 7. Bonus: show that global_indices map back to the original dataset
train_idx_global = data['train_idx']
global_idx = entry['global_indices']
print(f"Global indices map correctly: {np.array_equal(np.array(global_idx), train_idx_global[local_idx])}")


=== PHASE 3/4 SIMULATION ===
Experiment: synth_shift_vol10000_seed0
Boosting Branch (XGBoost) sees:    X shape (10000, 8), y shape (10000,)
Foundation Branch (TabPFN) sees:   X shape (10000, 8), y shape (10000,)
Are the training sets identical across branches? True
Validation set is fixed: True
ID Test set is fixed:    True
Global indices map correctly: True
